# MERT12 fixed-runtime final A100 QA

This diagnostic notebook reuses the immutable three trained heads and runs only packaged inference QA for the decoder-fixed ZIP. It writes fresh output directories and does not train or package.

In [ ]:
from pathlib import Path
import gc, hashlib, json, os, shutil, subprocess, sys, time
import torch
JOB=Path('/root/mert12_a100_20260918'); SOURCE=JOB/'source'; INPUT=JOB/'input'
OBJECT=Path('/root-1/mert12-a100-20260918'); OUTPUT=OBJECT/'output'; PACKAGE_ROOT=JOB/'final_package'
DEADLINE_EPOCH=float(os.environ['DEADLINE_EPOCH']); EXPECTED_DECODER='torchaudio.load+demucs.audio.convert_audio'
def sha(path):
    with path.open('rb') as stream: return hashlib.file_digest(stream,'sha256').hexdigest()
assert sys.version_info[:2]==(3,11) and torch.__version__=='2.7.1+cu128' and torch.version.cuda=='12.8'
assert time.time()<DEADLINE_EPOCH
package=OBJECT/'MERT12_full1322_submit.zip'; package_receipt=json.loads((OBJECT/'MERT12_full1322_submit.json').read_text())
assert package.stat().st_size==6268222843 and sha(package)=='e7bebbded6fef06517db0d42421a575a0440849248de99856796ffc3e5eb5ee5'
assert package_receipt['sha256']==sha(package) and package_receipt['crc_verified'] is True
assert package_receipt['choice']['demucs_input_decoder']==EXPECTED_DECODER
assert sha(OUTPUT/'mert12_full/head_seed41.pt')=='91f5bd17f786d5c9afccffb7516484cba6b695ba0156dce01c04a11aaca746d0'
assert sha(OUTPUT/'mert12_full/head_seed42.pt')=='a3dbec68027d686f6465d9144166741a330c80309a0b3538e78300d3d2990d6a'
assert sha(OUTPUT/'mert12_full/head_seed43.pt')=='a877808bd54326d51e0dedfc2331402a6f204f1534c8cc163db8036b5e895c9c'
for path in [PACKAGE_ROOT,*[OUTPUT/f'final_smoke_{name}' for name in ('original','permuted','subset')],OUTPUT/'final_supplemental_qa']:
    if path.exists(): shutil.rmtree(path)
shutil.unpack_archive(package,PACKAGE_ROOT)
assert (PACKAGE_ROOT/'script.py').is_file()
assert sha(SOURCE/'mert12_submission.py')=='30beefd173fbf10370c5dfa59e2e3f0ec9ee12b46a8ba4f24924a7d48e6f7f4e'
gc.collect(); torch.cuda.empty_cache(); assert torch.cuda.memory_allocated()==0
for name in ('original','permuted','subset'):
    target=OUTPUT/f'final_smoke_{name}'; target.mkdir()
    with (target/'runtime.log').open('w') as log:
        subprocess.run([sys.executable,str(PACKAGE_ROOT/'script.py'),'--test-dir',str(JOB/f'smoke_inputs/{name}/test'),'--sample-submission',str(JOB/f'smoke_inputs/{name}/sample_submission.csv'),'--output',str(target/'submission.csv'),'--device','cuda'],check=True,stdout=log,stderr=subprocess.STDOUT)
    runtime=json.loads((target/'runtime_receipt.json').read_text())
    assert runtime['device']=='cuda' and runtime['demucs_input_decoder']==EXPECTED_DECODER
subprocess.run([sys.executable,str(SOURCE/'mert12_submission.py'),'compare','--reference',str(OUTPUT/'final_smoke_original/submission.csv'),'--permuted',str(OUTPUT/'final_smoke_permuted/submission.csv'),'--subset',str(OUTPUT/'final_smoke_subset/submission.csv'),'--cached-reference',str(OUTPUT/'cached_mert12_reference.json'),'--output',str(OUTPUT/'final_submission_smoke.json')],check=True,cwd=SOURCE)
smoke=json.loads((OUTPUT/'final_submission_smoke.json').read_text())
assert smoke['max_probability_error']<=1e-7 and smoke['cached_raw_music_max_abs_error']<=1e-5
subprocess.run([sys.executable,str(SOURCE/'mert12_supplemental_qa.py'),'--package-root',str(PACKAGE_ROOT),'--source-audio',str(INPUT/'supplemental_real_source.wav'),'--source-sha256','f11541207692ae85606b95019ab828f2c554bdcd334759f7964c81793b63960e','--output',str(OUTPUT/'final_supplemental_qa'),'--deadline-epoch',str(DEADLINE_EPOCH),'--runtime-python',sys.executable],check=True,cwd=SOURCE)
supplemental=json.loads((OUTPUT/'final_supplemental_qa/supplemental_qa.json').read_text())
supp_runtime=json.loads((OUTPUT/'final_supplemental_qa/runtime_receipt.json').read_text())
assert supplemental['status']=='passed' and supp_runtime['device']=='cuda' and supp_runtime['demucs_input_decoder']==EXPECTED_DECODER
receipt={'schema_version':1,'status':'complete','package_sha256':sha(package),'package_bytes':package.stat().st_size,'decoder':EXPECTED_DECODER,'training_rerun':False,'package_reconstructed_on_gpu':True,'from_existing_archive':True,'checkpoint_sha256':['91f5bd17f786d5c9afccffb7516484cba6b695ba0156dce01c04a11aaca746d0','a3dbec68027d686f6465d9144166741a330c80309a0b3538e78300d3d2990d6a','a877808bd54326d51e0dedfc2331402a6f204f1534c8cc163db8036b5e895c9c'],'smoke':smoke,'supplemental_qa_sha256':sha(OUTPUT/'final_supplemental_qa/supplemental_qa.json'),'finished_at':time.time()}
(OUTPUT/'final_qa_receipt.json').write_text(json.dumps(receipt,indent=2,sort_keys=True)+'\n')
receipt